## Project: Scrape ICE Detention Facilities Website

**Description:** Scrape the ICE website for information about its detention facilities, including names and addresses.

**Goal:** Map the locations of ICE detention facilities to see how widely they are spread across the country. For example:

- Which state has the most ICE detention facilities?
- What does a comparison of border and interior states reveal?
- What other patterns emerge from the map?

1. Fetch page content

In [ ]:
base_url = 'https://www.ice.gov/detention-facilities'

In [ ]:
import requests

In [ ]:
headers= {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Safari/537.36'}
response = requests.get(base_url, headers=headers)

response.raise_for_status(), response.status_code

In [ ]:
response.text[:5000]

In [ ]:
# !pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup
content=BeautifulSoup(response.text, 'html.parser')

content

2. Extract the number of pages

In [ ]:
# pagination_content=content.find_all('nav', class_="usa-pagination")

In [ ]:
# for li in pagination_content:
#     lis=li.find_all('li')
#     for l in lis:
#         print(l)


In [ ]:
pagination_content=content.find_all("li", class_="usa-pagination__item usa-pagination__page-no")

In [ ]:
pages=[]
for li in pagination_content: 
    pages.append(li.get_text().strip())

pages

In [ ]:
page_nums=[]
for li in pagination_content:
    page_nums.append(li.find('a')['href'])
page_nums

In [ ]:
import numpy as np

np.arange(0, 8)

3. Scrape the first page

In [ ]:
grid_content=content.find_all('li', class_='grid')

In [ ]:
grid_content[0].find_all("span")

In [ ]:
grid_list = []
for grid in grid_content:
    facility_name = grid.find("span", class_="field-content").get_text().strip()
    address = grid.find('p', class_="address").get_text().strip()
    administrative_area = grid.find("span", class_="administrative-area").get_text().strip()
    postal_code = grid.find("span", class_="postal-code").get_text().strip()
    image_src = "https://www.ice.gov" + grid.find("img").get("src")

    grid_list.append({
        "facility_name": facility_name,
        "address": address,
        "administrative_area": administrative_area,
        "postal_code": postal_code,
        "image_src": image_src
    })


In [ ]:

import pandas as pd

df=pd.DataFrame(grid_list)
df.head()

In [ ]:
df.shape

4. Scrape all pages

In [ ]:
pages

In [ ]:
headers= {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Safari/537.36'}

dfs=[]
for page in pages:
    page_url = base_url + "?page=" + str(int(page) - 1)

    response = requests.get(page_url, headers=headers)

    print(page_url, response.status_code)

    grid_list = []
    for grid in grid_content:
        facility_name = grid.find("span", class_="field-content").get_text().strip()
        address = grid.find('p', class_="address").get_text().strip()
        administrative_area = grid.find("span", class_="administrative-area").get_text().strip()
        postal_code = grid.find("span", class_="postal-code").get_text().strip()
        image_src = "https://www.ice.gov" + grid.find("img").get("src")
        link_tag = grid.find("a")

        grid_list.append({
            "facility_name": facility_name,
            "address": address,
            "administrative_area": administrative_area,
            "postal_code": postal_code,
            "image_src": image_src,
             "link": "https://www.ice.gov" + link_tag["href"] if link_tag else None
        })

    
    df=pd.DataFrame(grid_list)
    # print(df.shape)
    dfs.append(df)


df=pd.concat(dfs, ignore_index=True)


df.shape

In [ ]:
import os

os.makedirs("scraped_data", exist_ok=True)

df.to_csv("scraped_data/ice_facilities.csv", index=False)

5. Check for errors

In [ ]:
df.head()

6. Scrape facility page

In [ ]:
## visit one of the pages and extract phone numbers

In [ ]:
import time 
import random

## write a function that can be applied to each row of the DataFrame

def get_info(url):
    try:
        ## try something
        time.sleep(random.uniform(1, 3))
    except Exception:
        return None


In [ ]:
## extract phone number from this 

7. Save save save

8. Read file

9. clean address


In [ ]:
## need to clean address before geocoding
import re

def clean_address(addr):
    # drop trailing country
   # "Natchez , MS" needs to be "Natchez, MS"

    ## remove extra spaces
    return 

In [ ]:
## clean address


8. Geocode

In [ ]:

# https://geocoding.geo.census.gov/geocoder/locations/onelineaddress

# https://nominatim.openstreetmap.org/search

In [ ]:
## Run either or geocoding


In [ ]:
## add lat, long, geo_source to the dataframe

9. Map it

In [ ]:
import plotly.express as px

mapped = df.dropna(subset=["lat", "lon"])  # skip rows that didn't geocode

fig = px.scatter_map(
    mapped,
    lat="lat",
    lon="lon",
    hover_name="name",
    hover_data={"clean_address": True, "lat": False, "lon": False},
    center={"lat": 39.8, "lon": -98.6},
    zoom=3,
    height=600,
)
fig.show()

In [ ]:

mapped = df.dropna(subset=["lat", "lon"])  # skip rows that didn't geocode

fig = px.scatter_geo(
    mapped,
    lat="lat",
    lon="lon",
    hover_name="name",
    hover_data={"clean_address": True, "lat": False, "lon": False},
    labels={"clean_address": "Address"},
    scope="usa",
    height=600,
)
fig.show()

10. Download Images

In [ ]:
os.makedirs("images", exist_ok=True)

In [ ]:
# Download images, add local path to the dataframe
    

11. do something fun with it

In [ ]:
import math
from PIL import Image, ImageOps

thumbnail_size = (300, 200)   # width, height of each frame/tile

# Load every image that downloaded successfully
frames = []
for path in df["image_file"].dropna():
    try:
        img = Image.open(path).convert("RGB")
        frames.append(ImageOps.fit(img, thumbnail_size))   # crop/resize to exactly thumbnail_size
    except Exception as e:
        print(f"Skipping {path}: {e}")

print(f"{len(frames)} images loaded")

In [ ]:
frames[0].save( # uses the first image as a starting frame and write the file
    "facilities.gif",
    save_all=True, # save multiple frames
    append_images=frames[1:], # adds the remaining images
    duration=300,   # ms per frame
    loop=0,         # 0 = loop forever, loop=1 would loop once
    optimize=True
)

In [ ]:
thumbnail_size = (300, 200)   # width, height of each frame/tile

cols = 10
rows = math.ceil(len(frames) / cols)

grid = Image.new("RGB", (cols * thumbnail_size[0], rows * thumbnail_size[1]), "white") # creates a blnak white canvas big enough for the whole grid 

for i, frame in enumerate(frames):
    x = (i % cols) * thumbnail_size[0] ## give the x-coordinate of the top-left corner of the tile
    y = (i // cols) * thumbnail_size[1] ## give the y-coordinate of the top-left corner of the tile
    grid.paste(frame, (x, y)) # pastes the frame onto the grid at the calculated position

grid.save("facilities_grid.jpg", quality=85)

-------------